In [19]:
import boto3
import torch 
import pandas as pd
import numpy as np
from io import BytesIO
from pathlib import Path
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
import matplotlib.pyplot as plt

devcie = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [20]:
#connect to tensor from s3
session = boto3.Session(profile_name="ATLAS")
s3 = session.client("s3", region_name="us-east-2")
BUCKET = "atlas-acm-research-f26"
TENSOR_PREFIX = "satellite/processed"
tensor_keys = []
paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(
    Bucket = BUCKET,
    Prefix = TENSOR_PREFIX
):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".pt"):
            tensor_keys.append(key)

tensor_keys.sort()
print('Total tensors found: ', len(tensor_keys))

Total tensors found:  250


In [21]:
#load im3 dataset and inspect available labels
df = pd.read_csv("./Data/im3_atlas_cleaned.csv",dtype={"id": str})
print('Columns: ', df.columns.to_list())
print('Total records: ', len(df))

if 'sqft' in df.columns:
    df['sqft'] = pd.to_numeric(df['sqft'], errors='coerce')
    print(df['sqft'].describe())
    print('Missing sqft: ', df['sqft'].isna().sum())


Columns:  ['id', 'sqft', 'lat', 'lon', 'type', 'size_class']
Total records:  1147
count    1.147000e+03
mean     4.715917e+05
std      1.971127e+06
min      2.040000e+02
25%      5.804350e+04
50%      1.323740e+05
75%      2.310440e+05
max      2.659752e+07
Name: sqft, dtype: float64
Missing sqft:  0


In [22]:
#calculating classification by quantiles to prevent imbalances
df["sqft"] = pd.to_numeric(df["sqft"], errors="coerce")

#quantiles
q1 = df["sqft"].quantile(0.25)
median = df["sqft"].median()
q3 = df["sqft"].quantile(0.75)

#assigning size labels
def classify_size(sqft):
    if pd.isna(sqft) or sqft <= 0:
        return None
    elif sqft < q1:
        return 0 #small
    elif sqft <= q3:
        return 1 #medium
    else:
        return 2 #large

df["size_label"] = df["sqft"].apply(classify_size)
class_names = ["Small", "Medium", "Large"]

print('\nClass distribution:')
for label, name in enumerate(class_names):
    count = (df["size_label"] == label).sum()
    print(f"{name}: {count}")



Class distribution:
Small: 287
Medium: 573
Large: 287


In [23]:
#get original data center ID from each tensor filename
tensor_df = pd.DataFrame({
    "id": [key.rsplit("/", 1)[-1].removesuffix(".pt") for key in tensor_keys],
    "s3_key": tensor_keys
})

#pair each tensor with its size label
training_df = tensor_df.merge(
    df[["id", "size_label"]],
    on="id",
    how="inner",
    validate="one_to_one"
)


In [24]:
#split into training, validation, testing

train_df, temp_df = train_test_split(
    training_df,
    test_size=0.30,
    random_state=42,
    stratify=training_df["size_label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["size_label"]
)

print('Training: ', len(train_df))
print('Validation: ', len(val_df))
print('Testing: ', len(test_df))

Training:  161
Validation:  35
Testing:  35


In [ ]:
#create PyTorch dataset and dataloaders
class DataCenterDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

        #normalization for ResNet
        self.normalize = transforms.Normalize(
            mean = [0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]

        response = s3.get_object(
            Bucket=BUCKET,
            Key=row["s3_key"]
        )

        data=torch.load(
            BytesIO(response["Body"].read()),
            map_location="cpu",
            weights_only=True
        )

        image = self.normalize(data["image"].float())
        label = int(row["size_label"])
        return image,label

train_loader = DataLoader(
    DataCenterDataset(train_df),
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    DataCenterDataset(val_df),
    batch_size=16
)

test_loader = DataLoader(
    DataCenterDataset(test_df),
    batch_size=16
)



In [31]:
#initializing resnet
model = models.resnet18(weights = models.ResNet18_Weights.DEFAULT)

#freeze pretrained layer
for param in model.parameters():
    param.requires_grad = False

#replace classfication layer iwht 3 output class
model.fc = nn.Linear(model.fc.in_features,3)
model = model.to(device)

#loss function
criterion = nn.CrossEntropyLoss()

#optimizer
optimizer = torch.optim.Adam(
    model.fc.parameters(), lr=0.001
)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/kevinlam/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:02<00:00, 23.4MB/s]


In [ ]:
#train resnet 
EPOCHS = 10

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(EPOCHS):
    #training
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    # validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    # calculate average loss and accuracy
    train_losses.append(train_loss / train_total)
    val_losses.append(val_loss / val_total)

    train_accuracies.append(train_correct / train_total)
    val_accuracies.append(val_correct / val_total)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_losses[-1]:.4f} | "
        f"Train Acc: {train_accuracies[-1]:.2%} | "
        f"Val Loss: {val_losses[-1]:.4f} | "
        f"Val Acc: {val_accuracies[-1]:.2%}"
    )

Epoch 1/10 | Train Loss: 1.0858 | Train Acc: 49.69% | Val Loss: 0.9357 | Val Acc: 60.00%
Epoch 2/10 | Train Loss: 1.0089 | Train Acc: 49.69% | Val Loss: 0.9436 | Val Acc: 54.29%
Epoch 3/10 | Train Loss: 0.9043 | Train Acc: 60.87% | Val Loss: 0.9206 | Val Acc: 65.71%
Epoch 4/10 | Train Loss: 0.9304 | Train Acc: 62.73% | Val Loss: 0.8938 | Val Acc: 60.00%
Epoch 5/10 | Train Loss: 0.9467 | Train Acc: 58.39% | Val Loss: 0.9313 | Val Acc: 60.00%
Epoch 6/10 | Train Loss: 0.9092 | Train Acc: 56.52% | Val Loss: 0.8872 | Val Acc: 60.00%
Epoch 7/10 | Train Loss: 0.8072 | Train Acc: 70.81% | Val Loss: 0.8935 | Val Acc: 62.86%
Epoch 8/10 | Train Loss: 0.7937 | Train Acc: 62.73% | Val Loss: 0.9067 | Val Acc: 57.14%
Epoch 9/10 | Train Loss: 0.7865 | Train Acc: 60.87% | Val Loss: 0.8653 | Val Acc: 68.57%
Epoch 10/10 | Train Loss: 0.8189 | Train Acc: 68.94% | Val Loss: 0.8592 | Val Acc: 62.86%


In [35]:
#performance evalutation

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.tolist())

print("Test Accuracy:",
      accuracy_score(all_labels, all_predictions))

#combines precision and recall - model performance across three classes
print("Macro F1:",
      f1_score(
          all_labels,
          all_predictions,
          average="macro",
          zero_division=0
      ))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2],
    target_names=class_names,
    zero_division=0
))

#what model predicted vs actual class - which size does model confuse
print("\nConfusion Matrix:")
print(confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2]
))

Test Accuracy: 0.5714285714285714
Macro F1: 0.38608776844070963

Classification Report:
              precision    recall  f1-score   support

       Small       1.00      0.17      0.29         6
      Medium       0.56      0.95      0.71        19
       Large       0.50      0.10      0.17        10

    accuracy                           0.57        35
   macro avg       0.69      0.40      0.39        35
weighted avg       0.62      0.57      0.48        35


Confusion Matrix:
[[ 1  5  0]
 [ 0 18  1]
 [ 0  9  1]]


## Performance evalution notes (first run)
The pretrained ResNet baseline has a strong majority class prediction bias with 16.7% recall for small, 94.7% for medium, and 10% for large classifications.

This suggest that the current model is not effectively classifying the facility size from the satellite image.

Possible reasons:
1. Class imbalance - Medium represents about 50% of the dataset since its split from 25-50-25 in quantiles.
2. Limited training images- This model on ran 35 images which is relatively small for a CNN to learn from
3. Frozen ResNet backboen - We only used the final classification layer for training limiting the ability to adapt to satellite imagery

In [ ]:
import boto3
import sagemaker
from sagemaker.pytorch import PyTorch
import os 
from dotenv import load_dotenv

load_dotenv()

boto_session = boto3.Session(profile_name="ATLAS", region_name="us-east-2")
sagemaker_session = sagemaker.Session(boto_session=boto_session)

role = os.getenv("SAGEMAKER_ROLE_ARN")

#configure the SageMaker training job.
estimator = PyTorch(
    entry_point="train_vision_baseline.py",
    role=role,
    instance_count=1,
    instance_type="ml.g4dn.xlarge",
    framework_version="2.2.0",
    py_version="py310",
    sagemaker_session=sagemaker_session,
    output_path="s3://atlas-acm-research-f26/artifacts/baseline/",
    base_job_name="atlas-vision-baseline",
    hyperparameters={
        "epochs": 50,
        "batch-size": 16,
        "learning-rate": 0.001,
    },
)

#start training and provide both inputs expected by train_vision_baseline.py.
estimator.fit(
    {
        "train": "s3://atlas-acm-research-f26/satellite/processed/",
        "metadata": "s3://atlas-acm-research-f26/dataset/im3_atlas_cleaned.csv",
    },
    wait=True,
    logs=True,
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: atlas-vision-baseline-2026-10-22-03-46-24-968
ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-training-job


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:30                                                                                   │
│                                                                                                  │
│   27 )                                                                                           │
│   28                                                                                             │
│   29 #start training and provide both inputs expected by train_vision_baseline.py.               │
│ ❱ 30 estimator.fit(                                                                              │
│   31 │   {                                                                                       │
│   32 │   │   "train": "s3://atlas-acm-research-f26/satellite/processed/",                        │
│   33 │   │   "metadata": "s3://atlas-acm-research-f26/dataset/im3_atlas_cleaned.csv",            │
│                                                                                                  │
│ /Users/kevinlam/ATLAS/.venv/lib/python3.13/site-packages/sagemaker/pytorch/estimator.py:833 in   │
│ fit                                                                                              │
│                                                                                                  │
│    830 │   │   │   │   │   "sagemaker_recipe_local_path": f"/opt/ml/input/data/{recipe_channel_  │
│    831 │   │   │   │   }                                                                         │
│    832 │   │   │   )                                                                             │
│ ❱  833 │   │   return super(PyTorch, self).fit(                                                  │
│    834 │   │   │   inputs=inputs,                                                                │
│    835 │   │   │   wait=wait,                                                                    │
│    836 │   │   │   logs=logs,                                                                    │
│                                                                                                  │
│ /Users/kevinlam/ATLAS/.venv/lib/python3.13/site-packages/sagemaker/telemetry/telemetry_logging.p │
│ y:171 in wrapper                                                                                 │
│                                                                                                  │
│   168 │   │   │   │   │   caught_ex = e                                                          │
│   169 │   │   │   │   finally:                                                                   │
│   170 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 171 │   │   │   │   │   │   raise caught_ex                                                    │
│   172 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   173 │   │   │   else:                                                                          │
│   174 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /Users/kevinlam/ATLAS/.venv/lib/python3.13/site-packages/sagemaker/telemetry/telemetry_logging.p │
│ y:142 in wrapper                                                                                 │
│                                                                                                  │
│   139 │   │   │   │   start_timer = perf_counter()                                               │
│   140 │   │   │   │   try:                                                                       │
│   141 │   │   │   │   │   # Call the original function                                           │
│ ❱ 142 │   │   │   │   │   response = func(*args, **kwargs) 